# TaaS Studio — Hypergraph Explorer integration validation

This notebook validates the integrated Studio page and the authenticated Discovery API using the **real TaaS DiscoveryEngine** with a local synthetic tool contract. It also runs the focused static/UI contract and Core projection regressions. The synthetic tool is never invoked; this is not an external MCP execution or production deployment test.


In [ ]:
from pathlib import Path
import subprocess, sys

REPO = "https://github.com/msellamiTN/taas-provenance-discovery.git"
PIN = "d3ab1247b40c6c1da76a9afaec42b36a911db69f"
WORKDIR = Path("/content/taas-provenance-discovery")
if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin"], check=True)
else:
    subprocess.run(["git", "clone", REPO, str(WORKDIR)], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "checkout", "--detach", PIN], check=True)
actual = subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "HEAD"], text=True).strip()
assert actual == PIN, (actual, PIN)
print("Pinned revision:", actual)
%cd /content/taas-provenance-discovery


In [ ]:
!pip -q install -e '.[test,studio]'
!python -m compileall -q src/taas


In [ ]:
!pytest -q tests/test_studio_hypergraph_explorer.py tests/test_studio_navigation_shell.py tests/test_studio_professional_ux.py tests/test_hypergraph_projection.py tests/test_engine_certificate_gate.py


## Integration coverage and validation boundary

- Serves the real Studio homepage and verifies the Explorer mount points.
- Calls the authenticated `/api/v1/discover` route with a local synthetic MCP-style contract.
- Uses the real `DiscoveryEngine` and checks that the returned Core projection contains the fired tool edge and reachable output type.
- Verifies missing bearer credentials are rejected and `execution_performed` remains false.

The synthetic contract is metadata only: no MCP server or tool is contacted. These tests exercise the FastAPI app in-process with `TestClient`; browser rendering, visual/keyboard behavior, remote deployment and production identity remain separate acceptance tests.
